# Post-benchmark ensemble and XGBoost research

The official TweetEval TEST benchmark was evaluated once and is frozen. This separate experiment uses TRAIN for all fitting and cross-validation and VALIDATION only for development comparison. Reusing TEST would turn the completed benchmark into a tuning set; this notebook contains no TEST prediction or evaluation code.

## Motivation and design

Frozen LR and LinearSVC disagree on 138 of 2,000 VALIDATION rows (6.90%). LR alone is correct on 56; SVM alone on 66. Their different errors motivate combination, but a two-model hard vote ties whenever they disagree. We explicitly resolve ties to LR, making hard voting identical to LR.

For soft voting, LinearSVC margins are not probabilities. Sigmoid CalibratedClassifierCV is fitted within TRAIN folds before mixing its probabilities with LR probabilities. Five weights are selected using three-fold stratified TRAIN CV. Calibration adds fitting cost and does not by itself guarantee better predictions.

For stacking, each meta-training row receives LR probabilities and SVM decision scores from base models that did not fit that row. Inner OOF fitting is nested inside three outer TRAIN folds for meta-parameter selection. The six meta-features feed a simple LogisticRegression classifier; full TF-IDF does not pass through.

## Sparse XGBoost design

XGBoost uses the same combined word (1,2) and character (3,5) TF-IDF representation. Vocabulary and IDF are fitted separately inside each TRAIN fold. All matrices stay scipy CSR sparse; the full feature matrix is never densified. Multiclass multi:softprob with mlogloss evaluates ten selected tree and sample-weight settings (30 fold fits), with macro-F1 as the selection score. Balanced weights, when used, are calculated from each fold-training label distribution.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
root = Path.cwd()
reports = root / 'reports' / 'experimental'
table = pd.read_csv(reports / 'ensemble_comparison.csv')
display(table.round(4))

,model,cv_macro_f1,cv_std,validation_accuracy,validation_macro_precision,validation_macro_recall,validation_macro_f1,negative_f1,neutral_f1,positive_f1,fit_seconds,predict_seconds
0,logistic_regression,0.6545,0.0021,0.6925,0.6642,0.6891,0.6728,0.5850,0.6861,0.7475,21.7286,0.3024
1,linear_svm,0.6517,0.0016,0.6975,0.6693,0.6771,0.6726,0.5710,0.6994,0.7473,12.3237,0.3007
2,hard_vote,0.6545,0.0021,0.6925,0.6642,0.6891,0.6728,0.5850,0.6861,0.7475,34.0523,0.6031
3,soft_vote,0.6558,0.0017,0.7015,0.6726,0.6845,0.6776,0.5787,0.7010,0.7531,53.1015,1.1962
4,stacking,0.6480,0.0019,0.7060,0.6916,0.6692,0.6781,0.5754,0.7109,0.7479,118.9627,0.6021
5,xgboost,0.5657,0.0011,0.5630,0.5569,0.5627,0.5412,0.4450,0.6087,0.5699,68.9684,0.1666


## Selected configurations and diagnostics

Soft voting selected LR weight 0.70 and SVM weight 0.30. Its TRAIN-CV macro-F1 was 0.6558; VALIDATION macro-F1 was 0.6776.

Stacking selected {'C': 1.0, 'class_weight': None}; TRAIN-CV macro-F1 0.6480; VALIDATION macro-F1 0.6781.

XGBoost candidate 8/10 selected {'max_depth': 5, 'learning_rate': 0.1, 'n_estimators': 40, 'subsample': 0.8, 'colsample_bytree': 0.5, 'min_child_weight': 3, 'reg_lambda': 1, 'sample_weight_policy': 'balanced'}; TRAIN-CV macro-F1 0.5657; VALIDATION macro-F1 0.5412. Tree feature importance is not interpreted causally.

In [2]:
weights = pd.read_csv(reports / 'soft_voting_weights.csv')
stacking = pd.read_csv(reports / 'stacking_meta_search.csv')
trees = pd.read_csv(reports / 'xgboost_results.csv')
display(weights.round(4))
display(stacking.round(4))
display(trees.sort_values('mean_cv_macro_f1', ascending=False).head(5).round(4))

,lr_weight,svm_weight,mean_cv_macro_f1,std_cv_macro_f1,fold_scores
0,0.3,0.7,0.6488,0.0025,"[0.6507708552558403, 0.6453308651950345, 0.650..."
1,0.4,0.6,0.6511,0.0023,"[0.6533960683739887, 0.6479018047041515, 0.652..."
2,0.5,0.5,0.6536,0.0023,"[0.656756012947359, 0.6514091017856902, 0.6527..."
3,0.6,0.4,0.6549,0.0012,"[0.6565818994514944, 0.6538586213322439, 0.654..."
4,0.7,0.3,0.6558,0.0017,"[0.6580896831577178, 0.6540270169362169, 0.655..."


,params,mean_cv_macro_f1,std_cv_macro_f1,fold_scores
0,"{""C"": 0.1, ""class_weight"": null}",0.6470,0.0016,"[0.6482775566863564, 0.6447657903834754, 0.648..."
1,"{""C"": 0.1, ""class_weight"": ""balanced""}",0.6462,0.0031,"[0.6505508670409622, 0.6445077053687548, 0.643..."
2,"{""C"": 1.0, ""class_weight"": null}",0.6480,0.0019,"[0.6492672744892343, 0.6453581667428222, 0.649..."
3,"{""C"": 1.0, ""class_weight"": ""balanced""}",0.6469,0.0029,"[0.6508654227206737, 0.6456719737704791, 0.644..."


,candidate,max_depth,learning_rate,n_estimators,subsample,colsample_bytree,min_child_weight,reg_lambda,sample_weight_policy,mean_cv_macro_f1,std_cv_macro_f1,fold_scores,total_cv_fit_seconds
7,7,5,0.10,40,0.8,0.50,3,1,balanced,0.5657,0.0011,"[0.5672825322254956, 0.5648221909929892, 0.564...",135.1507
9,9,7,0.03,80,0.8,0.50,3,5,balanced,0.5597,0.0016,"[0.5611002044225832, 0.557423234977196, 0.5605...",552.2791
3,3,5,0.05,60,0.8,0.75,1,5,balanced,0.5557,0.0020,"[0.555336982160013, 0.5534399618361489, 0.5582...",251.2015
5,5,3,0.10,50,0.8,0.75,3,5,balanced,0.5530,0.0011,"[0.5523500975666907, 0.5521399521919935, 0.554...",77.0387
1,1,3,0.05,40,0.8,0.50,1,1,balanced,0.5242,0.0024,"[0.5238384433131165, 0.5213768148115442, 0.527...",57.6687


## Holdout uncertainty and next decision

The best experimental model for this comparison is stacking. Its paired VALIDATION macro-F1 difference from frozen LR is +0.0052, with a seeded 2000-draw 95% percentile interval [-0.0109, +0.0211]. This describes sampling sensitivity on the same development holdout, not independent external evidence. Any promising model should be tested next on a newly collected, manually labeled brand-comment dataset. The official TweetEval TEST split remains closed.